# Rolling-Horizon HHCRSP Experiments

This notebook is the experiment-ready rolling-horizon implementation. It reproduces the 100-instance design used for the GA and Gurobi comparisons, records the decomposed objective, and validates every final schedule.

## 1. Configuration and data structures

Time is measured in minutes from 9:00 AM. Therefore, the dynamic-request cutoff at 300 minutes is 2:00 PM. The baseline rolling interval is 60 minutes; change only `delta` for the planned sensitivity analysis.

In [38]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import random
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from gurobipy import GRB, Model, quicksum


@dataclass(frozen=True)
class ExperimentConfig:
    instance_id: str
    num_patients: int
    num_nurses: int
    num_services: int
    dynamic_pct: float
    instance_seed: int
    sync_pct: float = 0.05
    delta: int = 60
    request_cutoff: int = 300  # Five hours after 9:00 AM = 2:00 PM
    shift_end: int = 480
    advance_window_width: int = 180
    cD: float = 1.0
    cT: float = 1.0
    cU: float = 1000.0
    time_limit: float | None = None


SERVICE_DURATIONS = {
    2: {1: 30, 2: 45},
    4: {1: 30, 2: 45, 3: 60, 4: 20},
    5: {1: 30, 2: 45, 3: 30, 4: 20, 5: 60},
}

RESULT_FIELDS = [
    "instance_id", "instance_seed", "num_patients", "num_nurses", "num_services",
    "dynamic_pct", "num_dynamic", "num_advance", "sync_pct", "num_sync", "delta",
    "fingerprint", "ZT", "Zd", "Zt", "Zu", "uncovered_total",
    "uncovered_advance", "uncovered_dynamic", "uncovered_patients",
    "runtime_seconds", "feasible", "validation_details", "reoptimizations",
    "solver_status_history", "error",
]




## 2. Matched experiment grid and instance generation

The random draws follow the original order: coordinates, requested service, synchronized patients, priority, advance time windows, and dynamic arrival times. Each generated instance receives a fingerprint for reproducibility checks.

In [39]:
def build_full_experiment_grid(delta: int = 60) -> list[ExperimentConfig]:
    """Reproduce the 100-instance design used in the original experiments."""
    families = {
        20: {"nurses": 4, "services": 2, "seeds": [27, 10, 42, 64, 73]},
        40: {"nurses": 8, "services": 4, "seeds": [5, 73, 20, 37, 42]},
        60: {"nurses": 10, "services": 5, "seeds": [10, 13, 27, 64, 42]},
        80: {"nurses": 12, "services": 4, "seeds": [73, 13, 27, 64, 42]},
    }
    dynamic_ratios = [0.10, 0.25, 0.50, 0.75, 0.90]
    configs = []
    for n, family in families.items():
        for seed_index, seed in enumerate(family["seeds"], start=1):
            for ratio_index, ratio in enumerate(dynamic_ratios, start=1):
                configs.append(
                    ExperimentConfig(
                        instance_id=f"{n}-{seed_index}-{ratio_index}",
                        num_patients=n,
                        num_nurses=family["nurses"],
                        num_services=family["services"],
                        dynamic_pct=ratio,
                        instance_seed=seed,
                        delta=delta,
                    )
                )
    return configs


def generate_instance(config: ExperimentConfig) -> dict:
    """Generate one matched instance using the original random-draw order."""
    if config.num_services not in SERVICE_DURATIONS:
        raise ValueError(f"No service-duration map for {config.num_services} services")
    if config.num_nurses % config.num_services != 0:
        raise ValueError("Caregivers must be divisible evenly across service types")

    rng = random.Random(config.instance_seed)
    patients = list(range(1, config.num_patients + 1))
    nurses = list(range(1, config.num_nurses + 1))
    services = list(range(1, config.num_services + 1))
    num_dynamic = int(config.dynamic_pct * config.num_patients)
    num_advance = config.num_patients - num_dynamic
    advance_patients = list(range(1, num_advance + 1))
    dynamic_patients = list(range(num_advance + 1, config.num_patients + 1))

    coords = {0: (25.0, 25.0)}
    for i in patients:
        coords[i] = (rng.uniform(0, 50), rng.uniform(0, 50))

    travel_time = {}
    for i in [0] + patients:
        for j in [0] + patients:
            if i != j:
                travel_time[i, j] = math.hypot(
                    coords[i][0] - coords[j][0],
                    coords[i][1] - coords[j][1],
                )

    request = {i: [rng.randint(1, config.num_services)] for i in patients}
    num_sync = int(config.sync_pct * config.num_patients)
    sync_patients = set(rng.sample(patients, num_sync))
    sync = {i: int(i in sync_patients) for i in patients}
    priority = {i: rng.randint(3, 5) for i in patients}

    patient_time_windows = {}
    for i in advance_patients:
        start = rng.randrange(0, 301, 15)
        patient_time_windows[i] = (start, start + config.advance_window_width)

    nurses_per_service = config.num_nurses // config.num_services
    qual = {}
    for s in services:
        first = (s - 1) * nurses_per_service
        qualified_nurses = set(nurses[first : first + nurses_per_service])
        for k in nurses:
            qual[k, s] = int(k in qualified_nurses)

    request_time = {
        i: rng.randint(0, config.request_cutoff) for i in dynamic_patients
    }

    fingerprint_payload = {
        "coords": coords,
        "request": request,
        "sync": sync,
        "priority": priority,
        "patient_time_windows": patient_time_windows,
        "request_time": request_time,
        "qual": {f"{k}-{s}": value for (k, s), value in qual.items()},
    }
    fingerprint = hashlib.sha256(
        json.dumps(fingerprint_payload, sort_keys=True).encode("utf-8")
    ).hexdigest()[:16]

    return {
        "config": config,
        "patients": patients,
        "nurses": nurses,
        "services": services,
        "advance_patients": advance_patients,
        "dynamic_patients": dynamic_patients,
        "coords": coords,
        "travel_time": travel_time,
        "service_duration": SERVICE_DURATIONS[config.num_services],
        "request": request,
        "sync_patients": sync_patients,
        "sync": sync,
        "priority": priority,
        "patient_time_windows": patient_time_windows,
        "nurse_time_windows": {k: (0, config.shift_end) for k in nurses},
        "qual": qual,
        "request_time": request_time,
        "fingerprint": fingerprint,
    }




## 3. Initial optimization and solution extraction

The initial model schedules only advance requests. A feasible incumbent is retained if a time limit is supplied.

In [40]:
def _configure_model(model: Model, config: ExperimentConfig) -> None:
    model.setParam("OutputFlag", 0)
    if config.time_limit is not None:
        model.setParam("TimeLimit", config.time_limit)


def _extract_solution(data: dict, z, S) -> tuple[dict, dict]:
    solution = {}
    routes = {k: [] for k in data["nurses"]}
    for i in z.keys():
        patient, nurse, service = i
        if z[i].X > 0.5:
            start = S[patient, nurse].X
            solution[patient, nurse, service] = start
            routes[nurse].append((start, patient, service))
    for nurse in routes:
        routes[nurse].sort()
    return solution, routes


def solve_initial_advance_problem(data: dict) -> dict:
    """Solve the advance-request problem available at the beginning of the day."""
    config = data["config"]
    patients = data["advance_patients"]
    nurses = data["nurses"]
    request = data["request"]
    travel = data["travel_time"]
    duration = data["service_duration"]
    nodes = [0] + patients
    arcs = [(i, j, k) for i in nodes for j in nodes if i != j for k in nurses]
    z_keys = [(i, k, request[i][0]) for i in patients for k in nurses]

    model = Model(f"RH_initial_{config.instance_id}")
    _configure_model(model, config)
    x = model.addVars(arcs, vtype=GRB.BINARY, name="x")
    z = model.addVars(z_keys, vtype=GRB.BINARY, name="z")
    S = model.addVars(patients, nurses, lb=0.0, vtype=GRB.CONTINUOUS, name="S")
    cover = model.addVars(
        [(i, request[i][0]) for i in patients], vtype=GRB.BINARY, name="cover"
    )

    travel_term = quicksum(travel[i, j] * x[i, j, k] for i, j, k in arcs)
    coverage_term = quicksum(
        data["priority"][i] * (1 - cover[i, request[i][0]]) for i in patients
    )
    model.setObjective(config.cT * travel_term + config.cU * coverage_term, GRB.MINIMIZE)
    M = 100000.0

    for k in nurses:
        model.addConstr(quicksum(x[0, j, k] for j in patients) <= 1)
        model.addConstr(quicksum(x[j, 0, k] for j in patients) <= 1)
        for j in nodes:
            model.addConstr(
                quicksum(x[i, j, k] for i in nodes if i != j)
                == quicksum(x[j, h, k] for h in nodes if h != j)
            )

    for i in patients:
        service = request[i][0]
        required = 1 + data["sync"][i]
        model.addConstr(quicksum(z[i, k, service] for k in nurses) == required * cover[i, service])
        for k in nurses:
            incoming = quicksum(x[j, i, k] for j in nodes if j != i)
            model.addConstr(z[i, k, service] == incoming)
            model.addConstr(z[i, k, service] <= data["qual"][k, service])
            earliest, latest = data["patient_time_windows"][i]
            model.addConstr(S[i, k] >= earliest * incoming)
            model.addConstr(S[i, k] <= latest * incoming)
            model.addConstr(S[i, k] >= travel[0, i] - M * (1 - x[0, i, k]))
            model.addConstr(
                S[i, k] + duration[service] + travel[i, 0]
                <= config.shift_end + M * (1 - x[i, 0, k])
            )

    for i in patients:
        service = request[i][0]
        for j in patients:
            if i == j:
                continue
            for k in nurses:
                model.addConstr(
                    S[i, k] + duration[service] + travel[i, j]
                    <= S[j, k] + M * (1 - x[i, j, k])
                )

    for i in patients:
        if not data["sync"][i]:
            continue
        service = request[i][0]
        for k1 in nurses:
            for k2 in nurses:
                if k1 >= k2:
                    continue
                model.addConstr(
                    S[i, k1] - S[i, k2]
                    <= M * (2 - z[i, k1, service] - z[i, k2, service])
                )
                model.addConstr(
                    S[i, k2] - S[i, k1]
                    <= M * (2 - z[i, k1, service] - z[i, k2, service])
                )

    model.optimize()
    has_solution = model.SolCount > 0
    solution, routes = _extract_solution(data, z, S) if has_solution else ({}, {k: [] for k in nurses})
    return {
        "has_solution": has_solution,
        "status": model.Status,
        "status_name": _status_name(model.Status),
        "solution": solution,
        "routes": routes,
        "objective": model.ObjVal if has_solution else None,
    }




## 4. Rolling-horizon state update and reoptimization

At every checkpoint, the code updates each caregiver's state. In the reoptimization model, the virtual start arc uses the caregiver's actual optimization anchor rather than the depot, so both travel feasibility and the travel objective use the correct first leg.

In [41]:
def determine_nurse_state(data: dict, solution: dict, routes: dict, current_time: float) -> dict:
    """Find each caregiver's state and future optimization anchor at a checkpoint."""
    duration = data["service_duration"]
    travel = data["travel_time"]
    states = {}
    globally_committed = set()

    for nurse in data["nurses"]:
        route = routes.get(nurse, [])
        anchor = 0
        available = current_time
        status = "available"
        committed = set()
        next_service = None

        for start, patient, service in route:
            end = start + duration[service]
            if current_time >= end - 1e-8:
                anchor = patient
                continue

            departure = start - travel.get((anchor, patient), 0.0)
            if current_time >= start - 1e-8:
                status = "serving"
                anchor = patient
                available = end
                committed.add(patient)
                globally_committed.add(patient)
            elif current_time >= departure - 1e-8:
                status = "traveling"
                anchor = patient
                available = end
                committed.add(patient)
                globally_committed.add(patient)
                next_service = (start, patient, service)
            else:
                next_service = (start, patient, service)
            break

        states[nurse] = {
            "status": status,
            "anchor_location": anchor,
            "available_time": available,
            "committed_patients": committed,
            "next_service": next_service,
        }

    # If one caregiver is already committed to a synchronized patient, lock all
    # caregivers assigned to that patient through completion. This is conservative
    # but prevents a reoptimized assignment from overlapping the preserved visit.
    for (patient, nurse, service), start in solution.items():
        if patient in globally_committed:
            end = start + duration[service]
            states[nurse]["committed_patients"].add(patient)
            states[nurse]["anchor_location"] = patient
            states[nurse]["available_time"] = max(states[nurse]["available_time"], end)
            if states[nurse]["status"] == "available":
                states[nurse]["status"] = "committed"

    return states


def classify_patients(
    data: dict,
    solution: dict,
    current_time: float,
    arrived_dynamic: set[int],
    nurse_state: dict,
) -> tuple[list[int], list[int], set[int], set[int]]:
    duration = data["service_duration"]
    assignments_by_patient = {}
    for (patient, nurse, service), start in solution.items():
        assignments_by_patient.setdefault(patient, []).append((start, service))

    completed = {
        patient
        for patient, assignments in assignments_by_patient.items()
        if assignments and all(current_time >= start + duration[service] - 1e-8 for start, service in assignments)
    }
    committed = set().union(
        *(state["committed_patients"] for state in nurse_state.values())
    )
    committed -= completed

    candidates = [
        i for i in data["advance_patients"] if i not in completed and i not in committed
    ]
    current_dynamic = []
    for i in sorted(arrived_dynamic):
        if i not in completed and i not in committed:
            candidates.append(i)
            current_dynamic.append(i)
    return candidates, current_dynamic, committed, completed


def solve_reoptimization(
    data: dict,
    candidates: list[int],
    current_dynamic: list[int],
    current_time: float,
    nurse_state: dict,
) -> dict:
    """Reoptimize flexible requests from each caregiver's current/future anchor."""
    config = data["config"]
    nurses = data["nurses"]
    request = data["request"]
    travel = data["travel_time"]
    duration = data["service_duration"]
    if not candidates:
        return {
            "has_solution": True,
            "status": GRB.OPTIMAL,
            "status_name": "OPTIMAL_EMPTY",
            "solution": {},
            "routes": {k: [] for k in nurses},
            "objective": 0.0,
        }

    nodes = [0] + candidates
    arcs = [(i, j, k) for i in nodes for j in nodes if i != j for k in nurses]
    z_keys = [(i, k, request[i][0]) for i in candidates for k in nurses]
    model = Model(f"RH_reopt_{config.instance_id}_{int(current_time)}")
    _configure_model(model, config)
    x = model.addVars(arcs, vtype=GRB.BINARY, name="x")
    z = model.addVars(z_keys, vtype=GRB.BINARY, name="z")
    S = model.addVars(candidates, nurses, lb=0.0, vtype=GRB.CONTINUOUS, name="S")
    du = model.addVars(current_dynamic, lb=0.0, vtype=GRB.CONTINUOUS, name="du")
    cover = model.addVars(
        [(i, request[i][0]) for i in candidates], vtype=GRB.BINARY, name="cover"
    )

    first_leg = quicksum(
        travel.get((nurse_state[k]["anchor_location"], j), 0.0) * x[0, j, k]
        for k in nurses
        for j in candidates
    )
    middle_legs = quicksum(
        travel[i, j] * x[i, j, k]
        for i in candidates
        for j in candidates
        if i != j
        for k in nurses
    )
    return_legs = quicksum(
        travel[i, 0] * x[i, 0, k] for i in candidates for k in nurses
    )
    lag_term = quicksum(du[i] for i in current_dynamic)
    coverage_term = quicksum(
        data["priority"][i] * (1 - cover[i, request[i][0]]) for i in candidates
    )
    model.setObjective(
        config.cD * lag_term
        + config.cT * (first_leg + middle_legs + return_legs)
        + config.cU * coverage_term,
        GRB.MINIMIZE,
    )
    M = 100000.0

    for k in nurses:
        model.addConstr(quicksum(x[0, j, k] for j in candidates) <= 1)
        model.addConstr(quicksum(x[j, 0, k] for j in candidates) <= 1)
        for j in nodes:
            model.addConstr(
                quicksum(x[i, j, k] for i in nodes if i != j)
                == quicksum(x[j, h, k] for h in nodes if h != j)
            )
        anchor = nurse_state[k]["anchor_location"]
        available = max(current_time, nurse_state[k]["available_time"])
        for j in candidates:
            model.addConstr(
                S[j, k] >= available + travel.get((anchor, j), 0.0) - M * (1 - x[0, j, k])
            )

    for i in candidates:
        service = request[i][0]
        required = 1 + data["sync"][i]
        model.addConstr(quicksum(z[i, k, service] for k in nurses) == required * cover[i, service])
        for k in nurses:
            incoming = quicksum(x[j, i, k] for j in nodes if j != i)
            model.addConstr(z[i, k, service] == incoming)
            model.addConstr(z[i, k, service] <= data["qual"][k, service])
            model.addConstr(S[i, k] <= config.shift_end * incoming)
            if i in data["advance_patients"]:
                earliest, latest = data["patient_time_windows"][i]
                model.addConstr(S[i, k] >= earliest * incoming)
                model.addConstr(S[i, k] <= latest * incoming)
            else:
                arrival = data["request_time"][i]
                model.addConstr(S[i, k] >= arrival * incoming)
                model.addConstr(du[i] >= S[i, k] - arrival - M * (1 - z[i, k, service]))
            model.addConstr(
                S[i, k] + duration[service] + travel[i, 0]
                <= config.shift_end + M * (1 - x[i, 0, k])
            )

    for i in candidates:
        service = request[i][0]
        for j in candidates:
            if i == j:
                continue
            for k in nurses:
                model.addConstr(
                    S[i, k] + duration[service] + travel[i, j]
                    <= S[j, k] + M * (1 - x[i, j, k])
                )

    for i in candidates:
        if not data["sync"][i]:
            continue
        service = request[i][0]
        for k1 in nurses:
            for k2 in nurses:
                if k1 >= k2:
                    continue
                model.addConstr(
                    S[i, k1] - S[i, k2]
                    <= M * (2 - z[i, k1, service] - z[i, k2, service])
                )
                model.addConstr(
                    S[i, k2] - S[i, k1]
                    <= M * (2 - z[i, k1, service] - z[i, k2, service])
                )

    model.optimize()
    has_solution = model.SolCount > 0
    solution, routes = _extract_solution(data, z, S) if has_solution else ({}, {k: [] for k in nurses})
    return {
        "has_solution": has_solution,
        "status": model.Status,
        "status_name": _status_name(model.Status),
        "solution": solution,
        "routes": routes,
        "objective": model.ObjVal if has_solution else None,
    }




## 5. Schedule display, objective calculation, and validation

The notebook displays a clean caregiver-by-caregiver schedule with route order, patient type, synchronization status, clock times, and travel from the previous location. The final objective is reported as (Z_T=Z_d+Z_t+Z_u). Delay is counted once per dynamic patient, including synchronized patients. The validator checks coverage multiplicity, qualifications, time windows, arrivals, synchronization, route feasibility, and return to depot.

In [42]:
def merge_solutions(
    data: dict,
    current_solution: dict,
    new_solution: dict,
    committed: set[int],
    completed: set[int],
) -> tuple[dict, dict]:
    preserved = committed | completed
    merged = {
        key: start for key, start in current_solution.items() if key[0] in preserved
    }
    for key, start in new_solution.items():
        if key[0] not in preserved:
            merged[key] = start
    routes = {k: [] for k in data["nurses"]}
    for (patient, nurse, service), start in merged.items():
        routes[nurse].append((start, patient, service))
    for nurse in routes:
        routes[nurse].sort()
    return merged, routes


def _clock_time(minutes_from_nine: float) -> str:
    total_minutes = int(round(9 * 60 + minutes_from_nine))
    hour_24 = (total_minutes // 60) % 24
    minute = total_minutes % 60
    suffix = "AM" if hour_24 < 12 else "PM"
    hour_12 = hour_24 % 12 or 12
    return f"{hour_12}:{minute:02d} {suffix}"


def schedule_dataframe(data: dict, solution: dict) -> pd.DataFrame:
    """Return a clean caregiver-by-caregiver schedule for display or export."""
    routes = {k: [] for k in data["nurses"]}
    for (patient, nurse, service), start in solution.items():
        routes[nurse].append((start, patient, service))
    for nurse in routes:
        routes[nurse].sort()

    rows = []
    for nurse in data["nurses"]:
        previous = 0
        for stop, (start, patient, service) in enumerate(routes[nurse], start=1):
            end = start + data["service_duration"][service]
            rows.append(
                {
                    "Caregiver": nurse,
                    "Stop": stop,
                    "Patient": patient,
                    "Service": service,
                    "Patient Type": "Advance" if patient in data["advance_patients"] else "Dynamic",
                    "Synchronized": "Yes" if data["sync"][patient] else "No",
                    "Start (min)": start,
                    "Start Time": _clock_time(start),
                    "End (min)": end,
                    "End Time": _clock_time(end),
                    "Travel from Previous": data["travel_time"].get((previous, patient), 0.0),
                }
            )
            previous = patient
    return pd.DataFrame(
        rows,
        columns=[
            "Caregiver", "Stop", "Patient", "Service", "Patient Type", "Synchronized",
            "Start (min)", "Start Time", "End (min)", "End Time", "Travel from Previous",
        ],
    )


def display_schedule(data: dict, solution: dict, title: str = "Schedule") -> pd.DataFrame:
    """Display the schedule neatly in Jupyter and return the underlying DataFrame."""
    frame = schedule_dataframe(data, solution)
    print(f"\n{title}")
    if frame.empty:
        print("No patients are scheduled.")
        return frame

    try:
        from IPython.display import display

        styled = (
            frame.style
            .hide(axis="index")
            .format(
                {
                    "Start (min)": "{:.1f}",
                    "End (min)": "{:.1f}",
                    "Travel from Previous": "{:.1f}",
                }
            )
            .set_caption(title)
            .set_table_styles(
                [
                    {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "14px")]},
                    {"selector": "th", "props": [("background-color", "#1F4E78"), ("color", "white"), ("text-align", "center")]},
                    {"selector": "td", "props": [("text-align", "center"), ("padding", "5px")]},
                ]
            )
        )
        display(styled)
    except (ImportError, NameError):
        print(frame.to_string(index=False, float_format=lambda value: f"{value:.1f}"))
    return frame


def compute_final_objective(data: dict, solution: dict) -> dict:
    """Calculate final objective components; synchronized delay is counted once."""
    config = data["config"]
    routes = {k: [] for k in data["nurses"]}
    starts_by_patient = {}
    covered_pairs = set()
    for (patient, nurse, service), start in solution.items():
        routes[nurse].append((start, patient, service))
        starts_by_patient.setdefault(patient, []).append(start)
        covered_pairs.add((patient, service))
    for nurse in routes:
        routes[nurse].sort()

    raw_travel = 0.0
    for nurse, route in routes.items():
        previous = 0
        for start, patient, service in route:
            raw_travel += data["travel_time"].get((previous, patient), 0.0)
            previous = patient
        if route:
            raw_travel += data["travel_time"].get((previous, 0), 0.0)

    raw_delay = 0.0
    for patient in data["dynamic_patients"]:
        if patient in starts_by_patient:
            raw_delay += max(0.0, min(starts_by_patient[patient]) - data["request_time"][patient])

    uncovered_pairs = []
    raw_uncovered_penalty = 0.0
    for patient in data["patients"]:
        for service in data["request"][patient]:
            if (patient, service) not in covered_pairs:
                uncovered_pairs.append((patient, service))
                raw_uncovered_penalty += data["priority"][patient]

    Zd = config.cD * raw_delay
    Zt = config.cT * raw_travel
    Zu = config.cU * raw_uncovered_penalty
    return {
        "ZT": Zd + Zt + Zu,
        "Zd": Zd,
        "Zt": Zt,
        "Zu": Zu,
        "raw_delay": raw_delay,
        "raw_travel": raw_travel,
        "raw_uncovered_penalty": raw_uncovered_penalty,
        "uncovered_pairs": uncovered_pairs,
        "uncovered_patients": sorted({patient for patient, service in uncovered_pairs}),
    }


def validate_final_schedule(data: dict, solution: dict, tolerance: float = 1e-5) -> dict:
    """Validate coverage, qualifications, timing, synchronization, and routes."""
    errors = []
    request = data["request"]
    duration = data["service_duration"]
    assignments = {}
    routes = {k: [] for k in data["nurses"]}

    for (patient, nurse, service), start in solution.items():
        if service not in request.get(patient, []):
            errors.append(f"Patient {patient}: unrequested service {service}")
        if not data["qual"].get((nurse, service), 0):
            errors.append(f"Nurse {nurse} is unqualified for service {service}")
        assignments.setdefault((patient, service), []).append((nurse, start))
        routes[nurse].append((start, patient, service))

        if patient in data["advance_patients"]:
            earliest, latest = data["patient_time_windows"][patient]
            if start < earliest - tolerance or start > latest + tolerance:
                errors.append(f"Patient {patient}: start {start:.3f} violates [{earliest}, {latest}]")
        else:
            arrival = data["request_time"][patient]
            if start < arrival - tolerance:
                errors.append(f"Dynamic patient {patient}: start {start:.3f} precedes arrival {arrival}")

    for patient in data["patients"]:
        for service in request[patient]:
            assigned = assignments.get((patient, service), [])
            required = 1 + data["sync"][patient]
            if len(assigned) not in (0, required):
                errors.append(
                    f"Patient {patient}, service {service}: {len(assigned)} caregivers; expected 0 or {required}"
                )
            if len({nurse for nurse, start in assigned}) != len(assigned):
                errors.append(f"Patient {patient}, service {service}: duplicate caregiver")
            if data["sync"][patient] and assigned:
                starts = [start for nurse, start in assigned]
                if max(starts) - min(starts) > tolerance:
                    errors.append(f"Synchronized patient {patient}: unequal starts {starts}")

    for nurse, route in routes.items():
        route.sort()
        previous = 0
        available = 0.0
        for start, patient, service in route:
            earliest_arrival = available + data["travel_time"].get((previous, patient), 0.0)
            if start < earliest_arrival - tolerance:
                errors.append(
                    f"Nurse {nurse}: cannot reach patient {patient} by {start:.3f}; earliest {earliest_arrival:.3f}"
                )
            available = start + duration[service]
            previous = patient
        if route:
            return_time = available + data["travel_time"].get((previous, 0), 0.0)
            if return_time > data["config"].shift_end + tolerance:
                errors.append(
                    f"Nurse {nurse}: depot return {return_time:.3f} exceeds {data['config'].shift_end}"
                )

    objective = compute_final_objective(data, solution)
    if abs(objective["ZT"] - objective["Zd"] - objective["Zt"] - objective["Zu"]) > tolerance:
        errors.append("Objective decomposition does not reconcile")
    return {"feasible": not errors, "errors": errors}




## 6. Experiment runners

The batch runner supports resuming. Each finished instance is appended immediately to the CSV, so completed results are retained if execution is interrupted.

In [43]:
def run_rolling_horizon(
    config: ExperimentConfig,
    verbose: bool = True,
    show_intermediate_schedules: bool = False,
) -> dict:
    start_clock = time.perf_counter()
    data = generate_instance(config)
    initial = solve_initial_advance_problem(data)
    if not initial["has_solution"]:
        return _failed_result(data, start_clock, f"Initial solve: {initial['status_name']}")

    solution = initial["solution"]
    routes = initial["routes"]
    if verbose and show_intermediate_schedules:
        display_schedule(data, solution, "Initial Advance-Request Schedule")
    arrived_dynamic = set()
    reoptimization_count = 0
    status_history = [f"initial:{initial['status_name']}"]

    for current_time in range(config.delta, config.shift_end + 1, config.delta):
        newly_arrived = {
            patient
            for patient in data["dynamic_patients"]
            if data["request_time"][patient] <= current_time and patient not in arrived_dynamic
        }
        arrived_dynamic |= newly_arrived
        nurse_state = determine_nurse_state(data, solution, routes, current_time)

        if verbose:
            state_text = ", ".join(
                f"N{k}:{state['status']}@{state['anchor_location']} until {state['available_time']:.1f}"
                for k, state in nurse_state.items()
            )
            print(
                f"[{config.instance_id}] t={current_time:3d}, new={sorted(newly_arrived)}, {state_text}"
            )

        if not newly_arrived:
            continue

        candidates, current_dynamic, committed, completed = classify_patients(
            data, solution, current_time, arrived_dynamic, nurse_state
        )
        reoptimized = solve_reoptimization(
            data, candidates, current_dynamic, current_time, nurse_state
        )
        status_history.append(f"t{current_time}:{reoptimized['status_name']}")
        if not reoptimized["has_solution"]:
            continue
        solution, routes = merge_solutions(
            data,
            solution,
            reoptimized["solution"],
            committed,
            completed,
        )
        reoptimization_count += 1
        if verbose and show_intermediate_schedules:
            display_schedule(
                data,
                solution,
                f"Updated Schedule after Checkpoint t={current_time}",
            )

    metrics = compute_final_objective(data, solution)
    validation = validate_final_schedule(data, solution)
    runtime = time.perf_counter() - start_clock
    uncovered_advance = len(
        [patient for patient in metrics["uncovered_patients"] if patient in data["advance_patients"]]
    )
    uncovered_dynamic = len(
        [patient for patient in metrics["uncovered_patients"] if patient in data["dynamic_patients"]]
    )
    result = {
        "instance_id": config.instance_id,
        "instance_seed": config.instance_seed,
        "num_patients": config.num_patients,
        "num_nurses": config.num_nurses,
        "num_services": config.num_services,
        "dynamic_pct": config.dynamic_pct,
        "num_dynamic": len(data["dynamic_patients"]),
        "num_advance": len(data["advance_patients"]),
        "sync_pct": config.sync_pct,
        "num_sync": len(data["sync_patients"]),
        "delta": config.delta,
        "fingerprint": data["fingerprint"],
        "ZT": metrics["ZT"],
        "Zd": metrics["Zd"],
        "Zt": metrics["Zt"],
        "Zu": metrics["Zu"],
        "uncovered_total": len(metrics["uncovered_patients"]),
        "uncovered_advance": uncovered_advance,
        "uncovered_dynamic": uncovered_dynamic,
        "uncovered_patients": json.dumps(metrics["uncovered_patients"]),
        "runtime_seconds": runtime,
        "feasible": validation["feasible"],
        "validation_details": " | ".join(validation["errors"]),
        "reoptimizations": reoptimization_count,
        "solver_status_history": " | ".join(status_history),
        "error": "",
    }
    if verbose:
        display_schedule(data, solution, "Final Rolling-Horizon Schedule")
        print(json.dumps(result, indent=2))
    return result


def run_experiments(
    configs: list[ExperimentConfig],
    output_csv: str | Path = "RH_results.csv",
    resume: bool = True,
    verbose: bool = False,
) -> list[dict]:
    """Run experiments and append each completed case immediately to CSV."""
    output_csv = Path(output_csv)
    completed = set()
    existing_results = []
    if resume and output_csv.exists():
        with output_csv.open("r", newline="", encoding="utf-8") as handle:
            existing_results = list(csv.DictReader(handle))
        completed = {row["instance_id"] for row in existing_results if row.get("instance_id")}

    new_results = []
    for index, config in enumerate(configs, start=1):
        if config.instance_id in completed:
            print(f"Skipping completed instance {config.instance_id}")
            continue
        print(f"Running {index}/{len(configs)}: {config.instance_id}")
        try:
            result = run_rolling_horizon(config, verbose=verbose)
        except Exception as exc:
            result = _blank_result(config, f"{type(exc).__name__}: {exc}")
        new_results.append(result)
        write_header = not output_csv.exists() or output_csv.stat().st_size == 0
        with output_csv.open("a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=RESULT_FIELDS, extrasaction="ignore")
            if write_header:
                writer.writeheader()
            writer.writerow(result)
    return existing_results + new_results


def _failed_result(data: dict, start_clock: float, message: str) -> dict:
    config = data["config"]
    result = _blank_result(config, message)
    result["fingerprint"] = data["fingerprint"]
    result["num_dynamic"] = len(data["dynamic_patients"])
    result["num_advance"] = len(data["advance_patients"])
    result["num_sync"] = len(data["sync_patients"])
    result["runtime_seconds"] = time.perf_counter() - start_clock
    return result


def _blank_result(config: ExperimentConfig, message: str) -> dict:
    result = {field: "" for field in RESULT_FIELDS}
    result.update(
        {
            "instance_id": config.instance_id,
            "instance_seed": config.instance_seed,
            "num_patients": config.num_patients,
            "num_nurses": config.num_nurses,
            "num_services": config.num_services,
            "dynamic_pct": config.dynamic_pct,
            "sync_pct": config.sync_pct,
            "delta": config.delta,
            "feasible": False,
            "validation_details": "",
            "error": message,
        }
    )
    return result


def _status_name(status: int) -> str:
    names = {
        GRB.LOADED: "LOADED",
        GRB.OPTIMAL: "OPTIMAL",
        GRB.INFEASIBLE: "INFEASIBLE",
        GRB.INF_OR_UNBD: "INF_OR_UNBD",
        GRB.UNBOUNDED: "UNBOUNDED",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.INTERRUPTED: "INTERRUPTED",
    }
    return names.get(status, f"STATUS_{status}")


## 7. Required one-instance test

Run this case first. It corresponds to the original 20-patient, seed-42, 25%-dynamic instance. Confirm that `feasible` is `True` and inspect any validation details before starting the batch.

In [44]:
test_config = ExperimentConfig(
    instance_id="20-1-1",
    num_patients=60,
    num_nurses=10,
    num_services=5,
    dynamic_pct=0.50,
    instance_seed=27,
    delta=60,
)

test_result = run_rolling_horizon(
    test_config,
    verbose=True,
    show_intermediate_schedules=True,
)
test_result


Initial Advance-Request Schedule


Caregiver,Stop,Patient,Service,Patient Type,Synchronized,Start (min),Start Time,End (min),End Time,Travel from Previous
1,1,5,1,Advance,Yes,22.3,9:22 AM,52.3,9:52 AM,22.3
1,2,19,1,Advance,No,120.0,11:00 AM,150.0,11:30 AM,14.2
1,3,24,1,Advance,No,300.0,2:00 PM,330.0,2:30 PM,12.5
1,4,15,1,Advance,No,338.5,2:38 PM,368.5,3:08 PM,8.5
2,1,5,1,Advance,Yes,22.3,9:22 AM,52.3,9:52 AM,22.3
2,2,10,1,Advance,No,76.3,10:16 AM,106.3,10:46 AM,24.0
2,3,27,1,Advance,No,126.3,11:06 AM,156.3,11:36 AM,19.9
2,4,17,1,Advance,No,195.0,12:15 PM,225.0,12:45 PM,3.4
2,5,3,1,Advance,No,237.4,12:57 PM,267.4,1:27 PM,12.4
3,1,25,2,Advance,No,15.0,9:15 AM,60.0,10:00 AM,10.1


[20-1-1] t= 60, new=[33, 34, 36, 41, 43, 45, 48, 53, 57], N1:available@5 until 60.0, N2:traveling@10 until 106.3, N3:available@25 until 60.0, N4:serving@22 until 78.7, N5:available@0 until 60.0, N6:available@0 until 60.0, N7:available@0 until 60.0, N8:available@6 until 60.0, N9:available@0 until 60.0, N10:available@0 until 60.0

Updated Schedule after Checkpoint t=60


Caregiver,Stop,Patient,Service,Patient Type,Synchronized,Start (min),Start Time,End (min),End Time,Travel from Previous
1,1,5,1,Advance,Yes,22.3,9:22 AM,52.3,9:52 AM,22.3
1,2,33,1,Dynamic,No,83.8,10:24 AM,113.8,10:54 AM,23.8
1,3,19,1,Advance,No,123.6,11:04 AM,153.6,11:34 AM,9.7
1,4,24,1,Advance,No,396.5,3:37 PM,426.5,4:07 PM,12.5
1,5,15,1,Advance,No,435.0,4:15 PM,465.0,4:45 PM,8.5
2,1,5,1,Advance,Yes,22.3,9:22 AM,52.3,9:52 AM,22.3
2,2,10,1,Advance,No,76.3,10:16 AM,106.3,10:46 AM,24.0
2,3,27,1,Advance,No,126.3,11:06 AM,156.3,11:36 AM,19.9
2,4,17,1,Advance,No,195.0,12:15 PM,225.0,12:45 PM,3.4
2,5,3,1,Advance,No,390.0,3:30 PM,420.0,4:00 PM,12.4


[20-1-1] t=120, new=[32, 35, 49, 50, 51, 52, 60], N1:traveling@19 until 153.6, N2:traveling@27 until 156.3, N3:traveling@48 until 170.9, N4:serving@11 until 124.9, N5:available@0 until 120.0, N6:serving@13 until 148.6, N7:available@57 until 120.0, N8:available@12 until 120.0, N9:serving@36 until 121.9, N10:serving@23 until 136.0

Updated Schedule after Checkpoint t=120


...

Caregiver,Stop,Patient,Service,Patient Type,Synchronized,Start (min),Start Time,End (min),End Time,Travel from Previous
1,1,5,1,Advance,Yes,22.3,9:22 AM,52.3,9:52 AM,22.3
1,2,33,1,Dynamic,No,83.8,10:24 AM,113.8,10:54 AM,23.8
1,3,19,1,Advance,No,123.6,11:04 AM,153.6,11:34 AM,9.7
1,4,50,1,Dynamic,No,173.1,11:53 AM,203.1,12:23 PM,19.5
1,5,38,1,Dynamic,Yes,242.6,1:03 PM,272.6,1:33 PM,39.5
1,6,3,1,Advance,No,280.6,1:41 PM,310.6,2:11 PM,8.0
1,7,17,1,Advance,No,323.0,2:23 PM,353.0,2:53 PM,12.4
2,1,5,1,Advance,Yes,22.3,9:22 AM,52.3,9:52 AM,22.3
2,2,10,1,Advance,No,76.3,10:16 AM,106.3,10:46 AM,24.0
2,3,27,1,Advance,No,126.3,11:06 AM,156.3,11:36 AM,19.9


[20-1-1] t=360, new=[], N1:available@17 until 360.0, N2:traveling@24 until 406.0, N3:serving@54 until 373.0, N4:serving@37 until 367.7, N5:available@20 until 360.0, N6:available@44 until 360.0, N7:traveling@26 until 410.0, N8:available@18 until 360.0, N9:serving@40 until 407.8, N10:serving@7 until 389.7
[20-1-1] t=420, new=[], N1:available@17 until 420.0, N2:available@24 until 420.0, N3:serving@21 until 450.2, N4:serving@16 until 459.2, N5:available@20 until 420.0, N6:available@44 until 420.0, N7:available@26 until 420.0, N8:available@18 until 420.0, N9:available@40 until 420.0, N10:serving@30 until 458.9
[20-1-1] t=480, new=[], N1:available@17 until 480.0, N2:available@15 until 480.0, N3:available@21 until 480.0, N4:available@16 until 480.0, N5:available@20 until 480.0, N6:available@44 until 480.0, N7:available@26 until 480.0, N8:available@18 until 480.0, N9:available@40 until 480.0, N10:available@30 until 480.0

Final Rolling-Horizon Schedule


Caregiver,Stop,Patient,Service,Patient Type,Synchronized,Start (min),Start Time,End (min),End Time,Travel from Previous
1,1,5,1,Advance,Yes,22.3,9:22 AM,52.3,9:52 AM,22.3
1,2,33,1,Dynamic,No,83.8,10:24 AM,113.8,10:54 AM,23.8
1,3,19,1,Advance,No,123.6,11:04 AM,153.6,11:34 AM,9.7
1,4,50,1,Dynamic,No,173.1,11:53 AM,203.1,12:23 PM,19.5
1,5,38,1,Dynamic,Yes,242.6,1:03 PM,272.6,1:33 PM,39.5
1,6,3,1,Advance,No,280.6,1:41 PM,310.6,2:11 PM,8.0
1,7,17,1,Advance,No,323.0,2:23 PM,353.0,2:53 PM,12.4
2,1,5,1,Advance,Yes,22.3,9:22 AM,52.3,9:52 AM,22.3
2,2,10,1,Advance,No,76.3,10:16 AM,106.3,10:46 AM,24.0
2,3,27,1,Advance,No,126.3,11:06 AM,156.3,11:36 AM,19.9


{
  "instance_id": "20-1-1",
  "instance_seed": 27,
  "num_patients": 60,
  "num_nurses": 10,
  "num_services": 5,
  "dynamic_pct": 0.5,
  "num_dynamic": 30,
  "num_advance": 30,
  "sync_pct": 0.05,
  "num_sync": 3,
  "delta": 60,
  "fingerprint": "b153bf329b449cde",
  "ZT": 18214.813983870412,
  "Zd": 2114.682296631916,
  "Zt": 1100.1316872384966,
  "Zu": 15000.0,
  "uncovered_total": 5,
  "uncovered_advance": 2,
  "uncovered_dynamic": 3,
  "uncovered_patients": "[2, 28, 39, 45, 53]",
  "runtime_seconds": 777.3743815000635,
  "feasible": true,
  "validation_details": "",
  "reoptimizations": 5,
  "solver_status_history": "initial:OPTIMAL | t60:OPTIMAL | t120:OPTIMAL | t180:OPTIMAL | t240:OPTIMAL | t300:OPTIMAL",
  "error": ""
}


{'instance_id': '20-1-1',
 'instance_seed': 27,
 'num_patients': 60,
 'num_nurses': 10,
 'num_services': 5,
 'dynamic_pct': 0.5,
 'num_dynamic': 30,
 'num_advance': 30,
 'sync_pct': 0.05,
 'num_sync': 3,
 'delta': 60,
 'fingerprint': 'b153bf329b449cde',
 'ZT': 18214.813983870412,
 'Zd': 2114.682296631916,
 'Zt': 1100.1316872384966,
 'Zu': 15000.0,
 'uncovered_total': 5,
 'uncovered_advance': 2,
 'uncovered_dynamic': 3,
 'uncovered_patients': '[2, 28, 39, 45, 53]',
 'runtime_seconds': 777.3743815000635,
 'feasible': True,
 'validation_details': '',
 'reoptimizations': 5,
 'solver_status_history': 'initial:OPTIMAL | t60:OPTIMAL | t120:OPTIMAL | t180:OPTIMAL | t240:OPTIMAL | t300:OPTIMAL',
 'error': ''}